In [1]:
import os
import pandas as pd
df_meteo = pd.read_csv("../data/MENSQ_33_previous-1950-2024.csv",
   sep=";")

colonnes_catnat = [
    "index",
    "code_geographique",
    "libelle_geographique",
    "epci",
    "libelle_epci",
    "departement",
    "libelle_departement",
    "region",
    "ept",
    "libelle_petr",
    "code_pnr",
    "libelle_pnr",
    "type_catastrophe",
    "date_debut",
    "date_fin",
    "date_arrete"
]

df_catnat = pd.read_csv(
    "../data/catnat.csv",
    sep=";",
    header=None,
    names=colonnes_catnat,
    encoding="utf-8",
    low_memory=False
)

# =====================================================================
# 0. PARAMÈTRES (les seuls endroits à modifier)
# =====================================================================
ANNEE_DEBUT = 1982          # CatNat démarre en 1982
ANNEE_FIN = 2022            # dernière année CatNat retenue, pour TOUTES les tables
                            # (2023-2024 sont souvent incomplets : voir le contrôle ci-dessous)
MIN_ANNEES_STATION = 40     # une station météo doit avoir au moins 40 années complètes
EPCI_CIBLE = "Bordeaux Métropole"
DOSSIER = "."               # dossier d'écriture des CSV (ex. "../notebook")


# =====================================================================
# 1. CATNAT : PRÉPARATION
# =====================================================================
# Code INSEE en texte sur 5 caractères (sinon "04054" devient 4054 et les jointures cassent)
df_catnat["code_commune"] = (df_catnat["code_geographique"].astype(str)
                             .str.replace(r"\.0$", "", regex=True).str.zfill(5))
df_catnat["departement"] = (df_catnat["departement"].astype(str)
                            .str.replace(r"\.0$", "", regex=True).str.zfill(2))

# Dates en vrai format date, et année = année de DÉBUT de l'événement
# (date_arrete est publiée avec retard, elle ne dit pas quand l'événement a eu lieu)
for c in ["date_debut", "date_fin", "date_arrete"]:
    df_catnat[c] = pd.to_datetime(df_catnat[c], errors="coerce")
df_catnat["annee"] = df_catnat["date_debut"].dt.year

# 8 types d'origine regroupés en 5 groupes lisibles
# (Sécheresse et Retrait-gonflement des argiles ne coexistent jamais la même année : même phénomène)
MAP = {"Sécheresse": "Sécheresse / argiles",
       "Retrait-gonflement des argiles": "Sécheresse / argiles",
       "Inondations": "Inondations",
       "Cyclones / Tempêtes": "Tempêtes",
       "Mouvements de terrain": "Mouvements de terrain"}
df_catnat["groupe"] = df_catnat["type_catastrophe"].map(MAP).fillna("Autres")

# ---- Contrôles (à lire avant de continuer) ----
print("Libellés EPCI contenant 'Bordeaux' :",
      df_catnat.loc[df_catnat["libelle_epci"].str.contains("Bordeaux", case=False, na=False),
                    "libelle_epci"].unique())
print("\nTypes classés dans 'Autres' :")
print(df_catnat.loc[df_catnat["groupe"] == "Autres", "type_catastrophe"].value_counts())
controle = df_catnat[df_catnat["libelle_epci"] == EPCI_CIBLE].dropna(subset=["annee"])
print("\nLignes CatNat de la métropole par année (8 dernières années) :")
print(controle.groupby(controle["annee"].astype(int)).size().tail(8))
# Si 2023-2024 ont beaucoup moins de lignes que les autres années, garde ANNEE_FIN = 2022.

# ---- Deux tables de travail ----
# cn = TOUTE la France, sur la période retenue (sert aux comparaisons)
cn = df_catnat[df_catnat["annee"].between(ANNEE_DEBUT, ANNEE_FIN)].copy()
cn["annee"] = cn["annee"].astype(int)
# bm = seulement les 28 communes de la métropole
bm = cn[cn["libelle_epci"] == EPCI_CIBLE].copy()
print("\nCommunes de la métropole :", bm["code_commune"].nunique())   # attendu : 28



Libellés EPCI contenant 'Bordeaux' : ['Bordeaux Métropole']

Types classés dans 'Autres' :
type_catastrophe
Grêle / neige    4229
Avalanche         131
Divers             65
Name: count, dtype: int64

Lignes CatNat de la métropole par année (8 dernières années) :
annee
2015     4
2017    18
2018    13
2020    21
2021    13
2022    38
2023     1
2024     6
dtype: int64

Communes de la métropole : 28


In [ ]:
# =====================================================================
# 2. TABLE episodes_par_annee  (Bordeaux Métropole, une ligne = une année)
# =====================================================================
# Un événement qui touche 28 communes = 28 lignes dans CatNat.
# On garde UNE ligne par épisode (même groupe, mêmes dates) pour qu'il compte pour 1.
episodes = bm.drop_duplicates(["groupe", "date_debut", "date_fin"])

# Nombre d'épisodes par année (lignes) et par groupe (colonnes)
ep_annee = (episodes.groupby(["annee", "groupe"]).size().unstack(fill_value=0)
            .reindex(range(ANNEE_DEBUT, ANNEE_FIN + 1), fill_value=0)   # années sans épisode = 0
            .rename_axis("annee"))
ep_annee.columns.name = None
ep_annee

In [3]:
# =====================================================================
# 3. TABLE arretes_par_commune  (France entière, une ligne = une commune)
# =====================================================================
# Un arrêté = une catastrophe naturelle reconnue officiellement pour une commune.
# Liste de TOUTES les communes de la base (même celles sans arrêté sur la période)
communes_ref = (df_catnat.drop_duplicates("code_commune")
                [["code_commune", "libelle_geographique", "departement",
                  "libelle_departement", "epci", "libelle_epci"]]
                .set_index("code_commune"))

nb_total = cn.groupby("code_commune").size().rename("nb_arretes")                  # total
nb_groupe = cn.groupby(["code_commune", "groupe"]).size().unstack(fill_value=0)    # par risque
nb_groupe.columns.name = None
colonnes_nb = list(nb_groupe.columns) + ["nb_arretes"]

communes_fr = communes_ref.join(nb_groupe).join(nb_total)
communes_fr[colonnes_nb] = communes_fr[colonnes_nb].fillna(0).astype(int)   # pas d'arrêté = 0
communes_fr = communes_fr.reset_index().sort_values("nb_arretes", ascending=False)

In [4]:
# =====================================================================
# 4. TABLE benchmark  (une ligne = un territoire, 3 niveaux)
# =====================================================================
# Pour chaque territoire : total d'arrêtés, nombre de communes, moyenne d'arrêtés PAR COMMUNE
# (la moyenne par commune rend comparables des territoires de tailles différentes)
def agreger(df, cle, niveau):
    g = (df.groupby(cle)
           .agg(nb_arretes=("nb_arretes", "sum"), nb_communes=("code_commune", "nunique"))
           .reset_index().rename(columns={cle: "territoire"}))
    g["arretes_par_commune"] = (g["nb_arretes"] / g["nb_communes"]).round(2)
    g.insert(0, "niveau", niveau)
    return g

dep = agreger(communes_fr, "libelle_departement", "Département")
met = agreger(communes_fr[communes_fr["libelle_epci"].str.contains("métropole", case=False, na=False)],
              "libelle_epci", "Métropole")
fr = pd.DataFrame([{"niveau": "France", "territoire": "France",
                    "nb_arretes": communes_fr["nb_arretes"].sum(),
                    "nb_communes": len(communes_fr),
                    "arretes_par_commune": round(communes_fr["nb_arretes"].sum() / len(communes_fr), 2)}])

benchmark = pd.concat([fr, dep, met], ignore_index=True).sort_values("arretes_par_commune",
                                                                      ascending=False)


In [5]:
# =====================================================================
# 5. TABLE climat_annuel  (Gironde, une ligne = une année)
# =====================================================================
# Colonnes météo retenues, et façon de les agréger sur l'année
# (somme pour les jours et la pluie, moyenne pour la température)
AGG = {"TX": "mean", "NBJTX30": "sum", "NBJTX35": "sum", "NBJTNS20": "sum",
       "RR": "sum", "ETP": "sum", "NBJRR30": "sum", "NBJRR50": "sum"}
cols = [c for c in AGG if c in df_meteo.columns]

for c in cols:
    df_meteo[c] = pd.to_numeric(df_meteo[c], errors="coerce")
df_meteo["annee"] = df_meteo["AAAAMM"].astype(int) // 100

# Pour une colonne : total (ou moyenne) annuel PAR STATION, en ne gardant que les années à 12 mois
# (un mois manquant fausserait la somme)
def annuel_par_station(col):
    g = df_meteo.groupby(["NUM_POSTE", "annee"])[col].agg([AGG[col], "count"]).reset_index()
    g = g.rename(columns={AGG[col]: col})
    return g[g["count"] == 12].drop(columns="count")

parts = {c: annuel_par_station(c) for c in cols}

# Stations « longues » : au moins 40 années complètes
# (évite qu'une station qui ouvre ou ferme crée une fausse tendance)
ref_col = "NBJTX30" if "NBJTX30" in parts else cols[0]
nb_ans = parts[ref_col].groupby("NUM_POSTE").size()
stations_ok = nb_ans[nb_ans >= MIN_ANNEES_STATION].index
print(f"\n{len(stations_ok)} stations retenues :")
print(df_meteo.loc[df_meteo["NUM_POSTE"].isin(stations_ok), ["NUM_POSTE", "NOM_USUEL"]]
      .drop_duplicates().to_string(index=False))

# Moyenne des stations retenues, année par année
meteo_annee = pd.concat(
    {c: p[p["NUM_POSTE"].isin(stations_ok)].groupby("annee")[c].mean() for c, p in parts.items()},
    axis=1).sort_index().rename_axis("annee")

# Nombre de stations qui contribuent chaque année (transparence)
meteo_annee["nb_stations"] = (parts[ref_col][parts[ref_col]["NUM_POSTE"].isin(stations_ok)]
                              .groupby("annee").size())

# Bilan hydrique = pluie - évapotranspiration (négatif = déficit en eau)
if "ETP" in meteo_annee and meteo_annee["ETP"].notna().any():
    meteo_annee["bilan_hydrique"] = meteo_annee["RR"] - meteo_annee["ETP"]

# Moyennes mobiles sur 10 ans pour lisser les courbes
for c in ["NBJTX30", "TX"]:
    if c in meteo_annee:
        meteo_annee[c + "_mm10"] = meteo_annee[c].rolling(10, min_periods=5).mean()


19 stations retenues :
 NUM_POSTE               NOM_USUEL
  33009001                ARCACHON
  33036001                   BAZAS
  33049001         BEYCHAC ET CAIL
  33063001         BORDEAUX-PAULIN
  33076001                   BUDOS
  33095001                CAPTIEUX
  33119001                   CENON
  33138001                 COUTRAS
  33214002                 LACANAU
  33236002              CAP-FERRET
  33281001       BORDEAUX-MERIGNAC
  33314005                PAUILLAC
  33484002          ST-SYMPHORIENJ
  33504001               SAUTERNES
  33522001                 TALENCE
  33529001                  CAZAUX
  33540001       VENDAYS-MONTALIVE
  33544001         LE VERDON-SUR-M
  33550001 VILLENAVE D'ORNON-INRAE


In [8]:
# =====================================================================
# 7. RECHARGEMENT POUR LE DASHBOARD (+ tables dérivées)
# =====================================================================
df_climat_annuel = pd.read_csv(os.path.join(DOSSIER, "climat_annuel.csv"))
df_ep = pd.read_csv(os.path.join(DOSSIER, "episodes_par_annee.csv"))
df_benchmark = pd.read_csv(os.path.join(DOSSIER, "benchmark.csv"))
# dtype=str sur les codes : sinon "04054" redevient 4054 à la relecture
df_communes = pd.read_csv(os.path.join(DOSSIER, "arretes_par_commune.csv"),
                          dtype={"code_commune": str, "departement": str})

# Dérivée 1 : heatmap = les 28 communes de la métropole (simple filtre, pas de fichier à part)
df_heatmap_comm = df_communes[df_communes["libelle_epci"] == EPCI_CIBLE].copy()

# Dérivée 2 : lien climat <-> catastrophes (une ligne par année, présente dans les deux tables)
df_lien_climat_catastrophe = df_climat_annuel.merge(df_ep, on="annee", how="inner")

In [9]:
# =====================================================================
# 6. EXPORT : 4 CSV
# =====================================================================
df_heatmap_comm.to_csv(os.path.join(DOSSIER, "heatmap_communes.csv"), index=False, encoding="utf-8-sig")
df_lien_climat_catastrophe.to_csv(os.path.join(DOSSIER, "lien_climat_catastrophes.csv"), index=False, encoding="utf-8-sig")
tables = {
    "climat_annuel": meteo_annee.reset_index(),
    "episodes_par_annee": ep_annee.reset_index(),
    "benchmark": benchmark,
    "arretes_par_commune": communes_fr,
}
for nom, t in tables.items():
    t.to_csv(os.path.join(DOSSIER, f"{nom}.csv"), index=False, encoding="utf-8-sig")
    print(f"{nom:22s} {t.shape}")


climat_annuel          (75, 13)
episodes_par_annee     (41, 6)
benchmark              (137, 5)
arretes_par_commune    (34598, 12)


In [ ]:
df_benchmark